In [1]:
# load necessary libraries
import json
import pandas as pd
import re
import sys
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
import umap

# set path to project root
base_path = Path.cwd() / "../"
sys.path.append(str(base_path.resolve()))

# load the ground truth data with augmentations
with open("../01_data/annotations/annotations_augmentations.json") as f:
    gt_augmentations = json.load(f)

In [2]:
# keep only texts with social groups and separate to have one sample per social group
filtered_gt = []
for sent in gt_augmentations:
    if sent["annotations"]:
        for ann in sent["annotations"]:
            filtered_gt.append(
                {
                    "sentence": sent["sentence"],
                    "group": ann["text"]
                }
            )

# compile two separate lists
sentence_list = []
mention_list = []

for sent in filtered_gt:
    mention_list.append(sent["group"])

In [4]:
class SpanDataset(Dataset):
    def __init__(self, tokenizer, mentions, max_len=64):
        self.dataset = []
        self.max_len = max_len

        for mention in mentions:
            encoding = tokenizer(
                mention,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            self.dataset.append({
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0)
            })

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]
    
    
    
class SpanEncoder(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # mean pooling
        h = outputs.last_hidden_state.mean(dim=1)
        return h

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
dataset = SpanDataset(tokenizer, mention_list)
dataloader = DataLoader(dataset)

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = SpanEncoder().to(device)

model.eval()

with torch.no_grad():
    all_embeddings = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        h = model(input_ids, attention_mask)
        all_embeddings.append(h.cpu())

all_embeddings = torch.cat(all_embeddings, dim=0).numpy()

In [8]:
dbscan = DBSCAN(eps=.1, min_samples=3, metric='cosine') 
labels = dbscan.fit_predict(all_embeddings)

cluster_dict = defaultdict(list)

for mention, label in zip(mention_list, labels):
    cluster_dict[label].append(mention)

for cluster_id, mentions in cluster_dict.items():
    print(f"Cluster {cluster_id} ({len(mentions)} mentions):")
    for mention in mentions:
        print(f"  - {mention}")
    print()

Cluster 0 (9 mentions):
  - judges
  - Judges
  - judges
  - judges
  - judges
  - judges
  - judges
  - judges
  - judges

Cluster 1 (1371 mentions):
  - young offenders
  - prisoners
  - family
  - family
  - offenders
  - the most disadvantaged
  - constituents
  - nurses
  - young people
  - pupils
  - Vulnerable young people
  - family
  - families
  - disadvantaged children
  - volunteers
  - young people
  - unpaid family carers
  - middle-grade emergency medicine doctors
  - constituents
  - young people
  - those who fail in their responsibilities
  - children living in poverty
  - young people under 25
  - parents
  - people in absolute poverty
  - the poorest 20% of families in this country
  - the families
  - unaccompanied children from Calais
  - disabled people
  - those with hidden disabilities
  - students
  - the refugees who have fled to neighbouring countries
  - older workers
  - political activists
  - disabled people
  - skilled workers
  - Toolmakers in my const